# Capstone: A Complete Causal Decoder

Connect embeddings, causal blocks, final normalization, logits, training, and generation.

## Before You Start

Course 10. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** scratch-07_evaluation, transformer-block.

## What You Will Be Able to Do

- Implement a full decoder forward pass
- Prove causal prefix invariance
- Train, save, restore, adapt, and evaluate a tiny model

## The tensor path from IDs to logits

Input IDs are [B,T]. Token lookup E[ids] and learned position lookup P[:T] are [B,T,D]. Add them, pass through pre-norm Transformer blocks, apply final layer normalization, and project to vocabulary logits [B,T,V]. This lab ties the output matrix to E: logits=hidden@E.T. Tied weights share parameters; they are not two independently trained copies.

Each block computes x <- x + attention(norm(x)), then x <- x + FFN(norm(x)). Attention projects Q,K,V, splits D into H heads of size D/H, masks future keys before softmax, mixes values, concatenates heads, and applies the output projection. The MLP uses the same documented GELU approximation as Course 1. Layer normalization is per token across features, with population variance and epsilon 1e-5. Learned scale and shift are omitted from this small browser contract.

## Train the same mechanism, then defend its limits

The notebook contains a small PyTorch implementation with autograd, an original synthetic sequence generator, disjoint train/validation/test sequences, fixed seeds, token-level CE, an optimizer loop, checkpoint continuation, and seeded autoregressive generation. It runs on CPU with no downloaded weights or data. Small vocabulary and width keep the exercise accessible; the resulting model is not a useful general-purpose assistant.

Causality is a behavioral invariant: modifying input token j must not change logits at positions earlier than j. Test multiple heads, multiple blocks, and context lengths. A shape-only test cannot establish this. To finish, record config, tokenizer/data identity, seed, parameter count, updates, train and held-out NLL, generated samples, and an explicit limitation. Do not select the best-looking seed and hide the rest.

## Worked Example: Trace a tied-output decoder

B=2, T=5, vocabulary=32, width=8, heads=2.

1. **Embed:** IDs [2,5] become hidden states [2,5,8]. Each head has width 4.
2. **Attend:** QK scores have shape [2,2,5,5]. The upper triangle is masked independently in each batch and head.
3. **Predict:** Final hidden [2,5,8] times E.T [8,32] gives logits [2,5,32]. Each position predicts the next token, not its own ID.

A runnable model is a chain of explicit contracts, not a collection of independently passing components.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Complete small GPT forward pass. NumPy; no autograd in the browser."""
import numpy as np

def _norm(x):
    return (x - x.mean(axis=-1, keepdims=True)) / np.sqrt(x.var(axis=-1, keepdims=True) + 1e-05)

def decoder_logits(ids, embedding, positions, blocks, heads):
    """Return [B,T,V] logits for a tied-output, pre-norm causal decoder.

ids integer [B,T], embedding [V,D], positions [max_T,D], T<=max_T.
D divisible by positive heads. blocks is a list of dictionaries:
q,k,v,o: [D,D]; up: [D,F]; down: [F,D]. No biases or learned norm affine.
Each block: residual attention(norm(x)); then residual MLP(norm(x)).
MLP uses GELU approximation h/(1+exp(-1.702*h)). Final norm, then @E.T.
Assume valid finite shapes/IDs. Do not modify any inputs or parameters."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

Vocabulary=40, width=12. Input and output weights are tied, excluding positions and blocks.

- Distinct token/output weight parameters

<details><summary>Worked answers</summary>

- 480: 40*12=480, not twice that count.

</details>

## Experiment

Run `experiments/scratch_training.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Run the CPU experiment end to end, including checkpoint equivalence and held-out loss. Report the actual values, not the worked-example numbers.
- This is a mechanism-scale training run. Large-scale data filtering, distributed efficiency, alignment safety, and production readiness remain separate courses.

## Transfer Problems

1. Replace learned positions with the RoPE mechanism from Course 2 and update the invariance tests.
2. Compare tied and untied output weights at a matched parameter or compute budget; state which quantity is held fixed.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[LLMs from Scratch: pretraining](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Complete small GPT forward pass. NumPy; no autograd in the browser."""
import numpy as np

def _norm(x):
    return (x - x.mean(axis=-1, keepdims=True)) / np.sqrt(x.var(axis=-1, keepdims=True) + 1e-05)

def decoder_logits(ids, embedding, positions, blocks, heads):
    """Return [B,T,V] logits for a tied-output, pre-norm causal decoder.

ids integer [B,T], embedding [V,D], positions [max_T,D], T<=max_T.
D divisible by positive heads. blocks is a list of dictionaries:
q,k,v,o: [D,D]; up: [D,F]; down: [F,D]. No biases or learned norm affine.
Each block: residual attention(norm(x)); then residual MLP(norm(x)).
MLP uses GELU approximation h/(1+exp(-1.702*h)). Final norm, then @E.T.
Assume valid finite shapes/IDs. Do not modify any inputs or parameters."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np

def _independent_decoder(ids, e, p, blocks, heads):
    def norm(v):
        return (v - np.mean(v)) / np.sqrt(np.mean((v - np.mean(v))**2) + 1e-5)
    outputs = []
    d = e.shape[1] // heads
    for row in ids:
        x = e[row] + p[:len(row)]
        for block in blocks:
            n = np.stack([norm(token) for token in x])
            q, k, v = [n @ block[name] for name in ('q', 'k', 'v')]
            contexts = []
            for t in range(len(row)):
                pieces = []
                for head in range(heads):
                    sl = slice(head*d, (head+1)*d)
                    scores = np.array([np.dot(q[t, sl], k[j, sl]) / np.sqrt(d) for j in range(t+1)])
                    probs = np.exp(scores-scores.max()); probs /= probs.sum()
                    pieces.append(sum(probs[j]*v[j, sl] for j in range(t+1)))
                contexts.append(np.concatenate(pieces))
            x = x + np.stack(contexts) @ block['o']
            h = np.stack([norm(token) for token in x]) @ block['up']
            x = x + (h / (1+np.exp(-1.702*h))) @ block['down']
        outputs.append(np.stack([norm(token) for token in x]) @ e.T)
    return np.stack(outputs)

rng = np.random.default_rng(281)
for width, heads, depth in [(4, 1, 0), (8, 2, 1), (6, 3, 2)]:
    ids = rng.integers(11, size=(2, 5))
    e, p = rng.normal(size=(11, width)), rng.normal(size=(7, width))
    blocks = [{**{name: rng.normal(size=(width, width))*.2 for name in ('q', 'k', 'v', 'o')}, 'up': rng.normal(size=(width, 9))*.2, 'down': rng.normal(size=(9, width))*.2} for _ in range(depth)]
    before = e.copy()
    result = decoder_logits(ids, e, p, blocks, heads)
    assert np.allclose(result, _independent_decoder(ids, e, p, blocks, heads), atol=1e-8), 'Full forward numerical contract'
    altered = ids.copy(); altered[:, -1] = (altered[:, -1]+1) % 11
    assert np.allclose(result[:, :-1], decoder_logits(altered, e, p, blocks, heads)[:, :-1]), 'No future information'
    assert np.allclose(result[:, :3], decoder_logits(ids[:, :3], e, p, blocks, heads)), 'Prefix invariance'
    assert np.array_equal(e, before)
print('Complete decoder checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Complete small GPT forward pass. NumPy; no autograd in the browser."""
import numpy as np


def _norm(x):
    return (x - x.mean(axis=-1, keepdims=True)) / np.sqrt(x.var(axis=-1, keepdims=True) + 1e-5)


def decoder_logits(ids, embedding, positions, blocks, heads):
    """Return [B,T,V] logits for a tied-output, pre-norm causal decoder.

    ids integer [B,T], embedding [V,D], positions [max_T,D], T<=max_T.
    D divisible by positive heads. blocks is a list of dictionaries:
    q,k,v,o: [D,D]; up: [D,F]; down: [F,D]. No biases or learned norm affine.
    Each block: residual attention(norm(x)); then residual MLP(norm(x)).
    MLP uses GELU approximation h/(1+exp(-1.702*h)). Final norm, then @E.T.
    Assume valid finite shapes/IDs. Do not modify any inputs or parameters.
    """
    batch, length = ids.shape
    width = embedding.shape[1]
    head_width = width // heads
    x = embedding[ids] + positions[:length]
    for block in blocks:
        n = _norm(x)
        q, k, v = [(n @ block[name]).reshape(batch, length, heads, head_width).transpose(0, 2, 1, 3) for name in ('q', 'k', 'v')]
        scores = (q @ k.swapaxes(-1, -2)) / np.sqrt(head_width)
        scores = np.where(np.tril(np.ones((length, length), bool)), scores, -np.inf)
        weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
        weights /= weights.sum(axis=-1, keepdims=True)
        context = (weights @ v).transpose(0, 2, 1, 3).reshape(batch, length, width)
        x = x + context @ block['o']
        h = _norm(x) @ block['up']
        # Stable sigmoid avoids overflow while preserving the Course 1 approximation.
        sigmoid = np.exp(-np.logaddexp(0., -1.702 * h))
        x = x + (h * sigmoid) @ block['down']
    return _norm(x) @ embedding.T


In [ ]:
import numpy as np

def _independent_decoder(ids, e, p, blocks, heads):
    def norm(v):
        return (v - np.mean(v)) / np.sqrt(np.mean((v - np.mean(v))**2) + 1e-5)
    outputs = []
    d = e.shape[1] // heads
    for row in ids:
        x = e[row] + p[:len(row)]
        for block in blocks:
            n = np.stack([norm(token) for token in x])
            q, k, v = [n @ block[name] for name in ('q', 'k', 'v')]
            contexts = []
            for t in range(len(row)):
                pieces = []
                for head in range(heads):
                    sl = slice(head*d, (head+1)*d)
                    scores = np.array([np.dot(q[t, sl], k[j, sl]) / np.sqrt(d) for j in range(t+1)])
                    probs = np.exp(scores-scores.max()); probs /= probs.sum()
                    pieces.append(sum(probs[j]*v[j, sl] for j in range(t+1)))
                contexts.append(np.concatenate(pieces))
            x = x + np.stack(contexts) @ block['o']
            h = np.stack([norm(token) for token in x]) @ block['up']
            x = x + (h / (1+np.exp(-1.702*h))) @ block['down']
        outputs.append(np.stack([norm(token) for token in x]) @ e.T)
    return np.stack(outputs)

rng = np.random.default_rng(281)
for width, heads, depth in [(4, 1, 0), (8, 2, 1), (6, 3, 2)]:
    ids = rng.integers(11, size=(2, 5))
    e, p = rng.normal(size=(11, width)), rng.normal(size=(7, width))
    blocks = [{**{name: rng.normal(size=(width, width))*.2 for name in ('q', 'k', 'v', 'o')}, 'up': rng.normal(size=(width, 9))*.2, 'down': rng.normal(size=(9, width))*.2} for _ in range(depth)]
    before = e.copy()
    result = decoder_logits(ids, e, p, blocks, heads)
    assert np.allclose(result, _independent_decoder(ids, e, p, blocks, heads), atol=1e-8), 'Full forward numerical contract'
    altered = ids.copy(); altered[:, -1] = (altered[:, -1]+1) % 11
    assert np.allclose(result[:, :-1], decoder_logits(altered, e, p, blocks, heads)[:, :-1]), 'No future information'
    assert np.allclose(result[:, :3], decoder_logits(ids[:, :3], e, p, blocks, heads)), 'Prefix invariance'
    assert np.array_equal(e, before)
print('Complete decoder checks passed.')


## Existing Course Model

The exact vendored mini_llm configuration and model are embedded here so the notebook does not depend on a repository clone or network import. The Torch model includes learned normalization affine parameters and exact GELU; the browser capstone deliberately omits affine parameters and uses its documented approximation.


In [ ]:
from dataclasses import asdict, dataclass
from typing import Dict, Any


@dataclass
class ModelConfig:
    vocab_size: int
    block_size: int = 64
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.1
    bias: bool = False

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if self.vocab_size <= 0 or self.block_size <= 0:
            raise ValueError("vocab_size and block_size must be positive")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

"""A bigram baseline and a small decoder-only Transformer."""
import math
from typing import Dict, Optional, Tuple
import torch
import torch.nn as nn
from torch.nn import functional as F

def sample_next_token(logits: torch.Tensor, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError('temperature must be greater than zero')
    logits = logits / temperature
    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        threshold = torch.topk(logits, top_k).values[:, [-1]]
        logits = logits.masked_fill(logits < threshold, float('-inf'))
    probabilities = F.softmax(logits, dim=-1)
    return torch.multinomial(probabilities, num_samples=1)

class BigramLanguageModel(nn.Module):
    """Each token directly stores logits for the token that follows it."""

    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)
        nn.init.zeros_(self.token_embedding.weight)
        self.vocab_size = vocab_size

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        logits = self.token_embedding(inputs)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), targets.reshape(-1))
        return (logits, loss)

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            (logits, _) = self(tokens[:, -1:])
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

class CausalSelfAttention(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.query_key_value = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.projection = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attention_dropout = nn.Dropout(config.dropout)
        self.residual_dropout = nn.Dropout(config.dropout)
        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer('causal_mask', causal_mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        (batch_size, time_steps, channels) = inputs.shape
        qkv = self.query_key_value(inputs)
        (query, key, value) = qkv.split(self.n_embd, dim=2)

        def split_heads(tensor: torch.Tensor) -> torch.Tensor:
            return tensor.view(batch_size, time_steps, self.n_head, self.head_dim).transpose(1, 2)
        (query, key, value) = map(split_heads, (query, key, value))
        attention = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)
        attention = attention.masked_fill(self.causal_mask[:, :, :time_steps, :time_steps] == 0, float('-inf'))
        weights = self.attention_dropout(F.softmax(attention, dim=-1))
        output = weights @ value
        output = output.transpose(1, 2).contiguous().view(batch_size, time_steps, channels)
        return self.residual_dropout(self.projection(output))

class FeedForward(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias), nn.GELU(), nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias), nn.Dropout(config.dropout))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)

class TransformerBlock(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.attention_norm = nn.LayerNorm(config.n_embd)
        self.attention = CausalSelfAttention(config)
        self.feed_forward_norm = nn.LayerNorm(config.n_embd)
        self.feed_forward = FeedForward(config)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        inputs = inputs + self.attention(self.attention_norm(inputs))
        inputs = inputs + self.feed_forward(self.feed_forward_norm(inputs))
        return inputs

class GPT(nn.Module):
    """A compact GPT-2-style decoder-only Transformer."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        (_, time_steps) = inputs.shape
        if time_steps > self.config.block_size:
            raise ValueError(f'Sequence length {time_steps} exceeds block size {self.config.block_size}')
        positions = torch.arange(time_steps, device=inputs.device)
        hidden = self.token_embedding(inputs) + self.position_embedding(positions)
        hidden = self.dropout(hidden)
        for block in self.blocks:
            hidden = block(hidden)
        logits = self.lm_head(self.final_norm(hidden))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.config.vocab_size), targets.reshape(-1))
        return (logits, loss)

    def parameter_count(self, exclude_position_embeddings: bool=True) -> int:
        count = sum((parameter.numel() for parameter in self.parameters()))
        if exclude_position_embeddings:
            count -= self.position_embedding.weight.numel()
        return count

    def optimizer_groups(self, weight_decay: float) -> list:
        (decay, no_decay) = ([], [])
        for parameter in self.parameters():
            (decay if parameter.dim() >= 2 else no_decay).append(parameter)
        return [{'params': decay, 'weight_decay': weight_decay}, {'params': no_decay, 'weight_decay': 0.0}]

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            context = tokens[:, -self.config.block_size:]
            (logits, _) = self(context)
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

def build_model(model_type: str, config: ModelConfig) -> nn.Module:
    if model_type == 'bigram':
        return BigramLanguageModel(config.vocab_size)
    if model_type == 'gpt':
        return GPT(config)
    raise ValueError("model_type must be 'bigram' or 'gpt'")

def checkpoint_model(checkpoint: Dict[str, object], device: torch.device) -> nn.Module:
    raw_config = checkpoint.get('model_config')
    model_type = checkpoint.get('model_type')
    if not isinstance(raw_config, dict) or not isinstance(model_type, str):
        raise ValueError('Checkpoint is missing model metadata')
    model = build_model(model_type, ModelConfig(**raw_config))
    model.load_state_dict(checkpoint['model_state'])
    return model.to(device)


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""CPU pretraining, token-weighted accumulation, and exact local resume.

Run: uv run --with numpy --with torch python experiments/scratch_training.py
The notebook embeds the existing mini_llm model source before this cell.
No downloaded data/weights. Fixed synthetic language, not natural-language skill.
"""
import copy
import hashlib
import io
import json
from pathlib import Path
import sys
import torch
from torch.nn import functional as F

if 'GPT' not in globals():
    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'colab/vendor/src'))
    from mini_llm.models import GPT
    from mini_llm.config import ModelConfig


def scratch_data(seed=19):
    """Independent unique sequences: next symbol usually advances by one."""
    generator = torch.Generator().manual_seed(seed)
    rows, seen = [], set()
    while len(rows) < 320:
        row = [int(torch.randint(8, (1,), generator=generator))]
        for _ in range(12):
            step = 1 if torch.rand((), generator=generator) < .82 else 3
            row.append((row[-1] + step) % 8)
        if tuple(row) not in seen:
            seen.add(tuple(row)); rows.append(row)
    data = torch.tensor(rows)
    permutation = torch.randperm(len(data), generator=generator)
    data = data[permutation]
    return data[:224], data[224:272], data[272:]


def scratch_pretrain(seed=7, updates=140):
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    config = ModelConfig(vocab_size=8, block_size=12, n_layer=1, n_head=2, n_embd=24, dropout=0.)
    model = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.006, weight_decay=.01)
    train, validation, test = scratch_data()
    generator = torch.Generator().manual_seed(seed + 1)

    def evaluate(rows):
        model.eval()
        with torch.no_grad():
            logits, _ = model(rows[:, :-1])
            return float(F.cross_entropy(logits.flatten(0, 1), rows[:, 1:].flatten()))

    initial = evaluate(validation)
    history = []
    for step in range(updates):
        model.train()
        indices = torch.randint(len(train), (24,), generator=generator)
        batch = train[indices]
        optimizer.zero_grad(set_to_none=True)
        # Unequal microbatch sizes still implement the full token-average gradient.
        tokens = batch[:, 1:].numel()
        for micro in (batch[:7], batch[7:]):
            logits, _ = model(micro[:, :-1])
            loss_sum = F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum')
            (loss_sum / tokens).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        optimizer.step()
        if (step + 1) % 35 == 0:
            history.append({'update': step + 1, 'train_nll': evaluate(train), 'validation_nll': evaluate(validation)})

    # Snapshot immediately before the next sampled update, including RNG state.
    checkpoint = copy.deepcopy(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
        rng=generator.get_state(), torch_rng=torch.get_rng_state(), step=updates, cursor=0,
        config=config.to_dict(), tokenizer_id='symbols-a-through-h-v1'))
    serialized = io.BytesIO()
    torch.save(checkpoint, serialized)
    serialized.seek(0)
    checkpoint = torch.load(serialized, weights_only=True)

    def next_update(m, opt, gen):
        m.train()
        batch = train[torch.randint(len(train), (24,), generator=gen)]
        opt.zero_grad(set_to_none=True)
        _, loss = m(batch[:, :-1], batch[:, 1:])
        loss.backward(); opt.step()
        return float(loss.detach())

    uninterrupted = next_update(model, optimizer, generator)
    restored = GPT(ModelConfig(**checkpoint['config']))
    restored.load_state_dict(checkpoint['model'])
    resumed_optimizer = torch.optim.AdamW(restored.parameters(), lr=.006, weight_decay=.01)
    resumed_optimizer.load_state_dict(checkpoint['optimizer'])
    resumed_rng = torch.Generator(); resumed_rng.set_state(checkpoint['rng'])
    torch.set_rng_state(checkpoint['torch_rng'])
    resumed = next_update(restored, resumed_optimizer, resumed_rng)
    assert uninterrupted == resumed
    assert all(torch.equal(a, b) for a, b in zip(model.parameters(), restored.parameters()))

    # Independent gradient-equivalence check before clipping/optimizer state.
    model.eval()
    batch = train[:9]
    optimizer.zero_grad(set_to_none=True)
    _, full_loss = model(batch[:, :-1], batch[:, 1:]); full_loss.backward()
    full_grads = [p.grad.clone() for p in model.parameters()]
    optimizer.zero_grad(set_to_none=True)
    for micro in (batch[:2], batch[2:]):
        logits, _ = model(micro[:, :-1])
        (F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum') / batch[:, 1:].numel()).backward()
    error = max(float((p.grad - g).abs().max()) for p, g in zip(model.parameters(), full_grads))
    assert error < 2e-6, error
    final = evaluate(validation)
    assert final < initial, 'This fixed toy run should learn its transition pattern'
    model.eval(); torch.manual_seed(seed + 100)
    sample = model.generate(torch.tensor([[0]]), 16, top_k=3).tolist()[0]
    report = dict(seed=seed, updates=updates, config=config.to_dict(), parameters=model.parameter_count(False),
        split_sizes=[len(train), len(validation), len(test)],
        data_sha256=hashlib.sha256(torch.cat([train, validation, test]).numpy().tobytes()).hexdigest(),
        initial_validation_nll=initial, final_validation_nll=final, test_nll=evaluate(test),
        accumulation_max_error=error, resume_exact=True, history=history, generated_ids=sample,
        tokenizer=dict(symbols=list('abcdefgh'),ids=list(range(8))),generated_text=''.join('abcdefgh'[i] for i in sample),
        limitation='Synthetic transition language; not a general-purpose LLM. Test used once after a fixed recipe.')
    return model, report


if __name__ == '__main__':
    _, report = scratch_pretrain()
    print(json.dumps(report, indent=2))
